In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print("train:", train.shape, "| test:", test.shape)
train.head()

In [ ]:
train.info() 

In [ ]:
print("Total missing values:", int(train.isna().sum().sum()))

train.describe().T.head(20) 

In [ ]:
counts = train["target"].value_counts().sort_index()

print(counts.to_dict())


fig, ax = plt.subplots()

ax.bar(counts.index.astype(str), counts.values, color="#4C72B0")

ax.set_title("Target Class Distribution")

ax.set_xlabel("Class"); ax.set_ylabel("Number of students")

plt.show() 

In [ ]:
sample_cols = ["nilai_minggu_01", "skor_motivasi", "aktivitas_hari_01",
   "tugas_selesai", "skor_tryout", "indeks_kehadiran"]


fig, axes = plt.subplots(2, 3, figsize=(13, 6))

for col, ax in zip(sample_cols, axes.ravel()):

    ax.hist(train[col], bins=30, color="#55A868", alpha=0.8)

    ax.set_title(col, fontsize=10)

plt.tight_layout(); plt.show() 

In [ ]:
feat_cols = [col for col in train.columns if col not in ("id", "target")]


corr = (train[feat_cols]

        .apply(lambda col: np.corrcoef(col, train["target"])[0, 1])

        .abs()

        .sort_values(ascending=False))


print("Max |correlation|   :", round(corr.max(), 3))

print("Median |correlation|:", round(corr.median(), 3))


fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(range(len(corr)), corr.values, color="#55A868")

ax.axhline(0.05, color="red", ls="--", lw=1, label="0.05 threshold")

ax.set_xticks(range(len(corr)))

ax.set_xticklabels(corr.index, rotation=90, fontsize=7)

ax.set_title("Absolute Correlation of Each Raw Column with Target")

ax.set_ylabel("|correlation|"); ax.legend()

plt.tight_layout(); plt.show() 

In [ ]:
subset = (["nilai_minggu_01", "nilai_minggu_06", "nilai_minggu_12"] +

          ["aktivitas_hari_01", "aktivitas_hari_08", "aktivitas_hari_16"] +

          ["tugas_selesai", "tugas_diberikan", "skor_tryout"])

cmat = train[subset].corr()


fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(cmat, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(subset))); ax.set_xticklabels(subset, rotation=90, fontsize=8)

ax.set_yticks(range(len(subset))); ax.set_yticklabels(subset, fontsize=8)

fig.colorbar(im, fraction=0.046, pad=0.04)

ax.set_title("Correlation Heatmap (column subset)")

plt.tight_layout(); plt.show() 

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline

from sklearn.model_selection import cross_val_score

y = train["target"]
X_raw = train[feat_cols]
def evaluate(features, target=y, cv=5):

    '''Mean CV accuracy, standardized Logistic Regression.'''

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))

    return cross_val_score(model, features, target, cv=cv).mean()

baseline = evaluate(X_raw)

print(f"Baseline (raw columns, Logistic Regression): {baseline:.3f}") 

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(tree_method="hist", n_estimators=400, max_depth=6,

                    learning_rate=0.1, random_state=0, n_jobs=-1,

                    eval_metric="mlogloss", verbosity=0)

xgb_raw = cross_val_score(xgb, X_raw, y, cv=3).mean()

print(f"XGBoost (raw columns): {xgb_raw:.3f}") 

In [ ]:
week_cols = [f"nilai_minggu_{i:02d}" for i in range(1, 13)]
day_cols  = [f"aktivitas_hari_{i:02d}" for i in range(1, 17)]
fe = train[feat_cols].copy()

for name, cols in [("week", week_cols), ("day", day_cols)]:

    fe[f"{name}_mean"]  = train[cols].mean(axis=1)
    fe[f"{name}_std"]   = train[cols].std(axis=1)
    fe[f"{name}_max"]   = train[cols].max(axis=1)
    fe[f"{name}_min"]   = train[cols].min(axis=1)
    fe[f"{name}_range"] = train[cols].max(axis=1) - train[cols].min(axis=1)

added = [c for c in fe.columns if c not in feat_cols]
print("Added summary features:", added)
acc_agg = evaluate(fe)
print(f"\nRaw baseline           : {baseline:.3f}")
print(f"+ group aggregations   : {acc_agg:.3f}")



In [ ]:
agg_corr = (fe[added]

            .apply(lambda col: abs(np.corrcoef(col, y)[0, 1]))
            .sort_values(ascending=False)) 

In [ ]:
acc_agg_xgb = cross_val_score(xgb, fe, y, cv=3).mean()
print(f"XGBoost raw columns     : {xgb_raw:.3f}")
print(f"XGBoost + aggregations  : {acc_agg_xgb:.3f}")
print(agg_corr.round(3).to_string())

fig, ax = plt.subplots()
ax.barh(agg_corr.index[::-1], agg_corr.values[::-1], color="#C44E52")
ax.set_title("Correlation of Aggregated Features with Target")
ax.set_xlabel("|correlation|")
plt.tight_layout(); plt.show()

In [ ]:
summary = pd.DataFrame([
    ("Linear - raw columns", baseline),
    ("Linear - + aggregations", acc_agg),
    ("XGBoost - raw columns", xgb_raw),
    ("XGBoost - + aggregations", acc_agg_xgb),
], columns=["pipeline", "cv_accuracy"])
print(summary.to_string(index=False)) 

In [ ]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    tree_method="hist", 
    n_estimators=500, 
    max_depth=5,
    learning_rate=0.03, 
    random_state=0, 
    n_jobs=-1,
    eval_metric="mlogloss"
)
xgb_model.fit(X_train_final, y)
print("Model XGBoost berhasil dilatih!")

In [ ]:
pred_final = xgb_model.predict(X_test_final)
submission_final = pd.DataFrame({
    "id": test["id"], 
    "target": pred_final
})
submission_final.to_csv("submission_final.csv", index=False)
print("submission_final.csv sukses disimpan!")
submission_final.head()

In [ ]:
def make_features_engineered(df):
    out = df[[c for c in df.columns if c not in ("id", "target")]].copy()
    wk = [f"nilai_minggu_{i:02d}" for i in range(1, 13)]
    dy = [f"aktivitas_hari_{i:02d}" for i in range(1, 17)]
    
    for name, cols in [("week", wk), ("day", dy)]:
        out[f"{name}_mean"]  = df[cols].mean(axis=1)
        out[f"{name}_std"]   = df[cols].std(axis=1)
        out[f"{name}_max"]   = df[cols].max(axis=1)
        out[f"{name}_min"]   = df[cols].min(axis=1)
        out[f"{name}_range"] = df[cols].max(axis=1) - df[cols].min(axis=1)
        
    out['tren_nilai_total'] = df['nilai_minggu_12'] - df['nilai_minggu_01']
    out['tren_nilai_akhir'] = df['nilai_minggu_12'] - df['nilai_minggu_08']
    out['rasio_tugas'] = df['tugas_selesai'] / (df['tugas_diberikan'] + 1e-5)
    out['total_beban_belajar'] = df['tugas_diberikan'] * out['day_mean']
    out['interaksi_kehadiran_motivasi'] = df['indeks_kehadiran'] * df['skor_motivasi']
    
    out['kehadiran_per_motivasi'] = df['indeks_kehadiran'] / (df['skor_motivasi'] + 1e-5)
    out['day_cv'] = out['day_std'] / (out['day_mean'] + 1e-5)
    out['tryout_per_beban'] = df['skor_tryout'] / (df['tugas_diberikan'] + 1e-5)
    
    return out
X_train_final = make_features_engineered(train)
X_test_final  = make_features_engineered(test)
print("Jumlah kolom setelah upgrade:", X_train_final.shape[1])

Jumlah kolom setelah upgrade: 59


In [ ]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    tree_method="hist",
    n_estimators=1000,          
    max_depth=4,                
    learning_rate=0.015,        
    subsample=0.8,              
    colsample_bytree=0.8,       
    random_state=0,
    n_jobs=-1,
    eval_metric="mlogloss"
)
xgb_model.fit(X_train_final, y)
print("Model XGBoost Versi Upgrade Berhasil Dilatih!")

Model XGBoost Versi Upgrade Berhasil Dilatih!


In [ ]:
pred_final = xgb_model.predict(X_test_final)
submission_final = pd.DataFrame({
    "id": test["id"], 
    "target": pred_final
})
submission_final.to_csv("submission_final.csv", index=False)
print("submission_final.csv sukses disimpan!")
submission_final.head()

submission_final.csv sukses disimpan!


,id,target
0,3,2
1,12,2
2,14,3
3,18,3
4,28,0


In [ ]:
from sklearn.model_selection import cross_val_score
skor_baru = cross_val_score(xgb_model, X_train_final, y, cv=3).mean()
print(f"Akurasi Model Baru Setelah Upgrade: {skor_baru * 100:.2f}%")

Akurasi Model Baru Setelah Upgrade: 46.03%


In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score
cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.03,
    depth=5,
    random_seed=42,
    verbose=0 # Biar gak menuh-menuhin log
)

skor_cat = cross_val_score(cat_model, X_train_final, y, cv=3).mean()
print(f"Akurasi pake CatBoost: {skor_cat * 100:.2f}%")

Akurasi pake CatBoost: 51.31%


In [43]:
cat_model.fit(X_train_final, y)
pred_cat = cat_model.predict(X_test_final)
submission_cat = pd.DataFrame({
    "id": test["id"], 
    "target": pred_cat.flatten()
})
submission_cat.to_csv("submission_final.csv", index=False)
print("submission_final.csv BERHASIL DI-UPGRADE PAKE CATBOOST (51.31%)!")
submission_cat.head()

submission_final.csv BERHASIL DI-UPGRADE PAKE CATBOOST (51.31%)!


,id,target
0,3,2
1,12,2
2,14,3
3,18,3
4,28,0


In [ ]:
xgb_model.fit(X_train_final, y)
proba_cat = cat_model.predict_proba(X_test_final)
proba_xgb = xgb_model.predict_proba(X_test_final)
proba_ensemble = (0.7 * proba_cat) + (0.3 * proba_xgb)
pred_ensemble = proba_ensemble.argmax(axis=1)
submission_ensemble = pd.DataFrame({
    "id": test["id"], 
    "target": pred_ensemble
})
submission_ensemble.to_csv("submission_final.csv", index=False)
print("Submission final hasil kolaborasi Ensemble sukses disimpan!")
submission_ensemble.head()

Submission final hasil kolaborasi Ensemble sukses disimpan!


,id,target
0,3,2
1,12,2
2,14,3
3,18,3
4,28,0


In [48]:
def make_features_ultra(df):
    out = df[[c for c in df.columns if c not in ("id", "target")]].copy()
    wk = [f"nilai_minggu_{i:02d}" for i in range(1, 13)]
    dy = [f"aktivitas_hari_{i:02d}" for i in range(1, 17)]
    
    for name, cols in [("week", wk), ("day", dy)]:
        out[f"{name}_mean"]  = df[cols].mean(axis=1)
        out[f"{name}_std"]   = df[cols].std(axis=1)
        out[f"{name}_max"]   = df[cols].max(axis=1)
        out[f"{name}_min"]   = df[cols].min(axis=1)
        out[f"{name}_range"] = df[cols].max(axis=1) - df[cols].min(axis=1)
        
    out['tren_nilai_total'] = df['nilai_minggu_12'] - df['nilai_minggu_01']
    out['tren_nilai_akhir'] = df['nilai_minggu_12'] - df['nilai_minggu_08']
    out['rasio_tugas'] = df['tugas_selesai'] / (df['tugas_diberikan'] + 1e-5)
    out['total_beban_belajar'] = df['tugas_diberikan'] * out['day_mean']
    out['interaksi_kehadiran_motivasi'] = df['indeks_kehadiran'] * df['skor_motivasi']
    
    out['kehadiran_per_motivasi'] = df['indeks_kehadiran'] / (df['skor_motivasi'] + 1e-5)
    out['day_cv'] = out['day_std'] / (out['day_mean'] + 1e-5)
    out['tryout_per_beban'] = df['skor_tryout'] / (df['tugas_diberikan'] + 1e-5)
    
    out['stabilitas_nilai'] = out['week_std'] / (out['week_mean'] + 1e-5)
    out['rasio_literasi_tryout'] = df['skor_literasi'] / (df['skor_tryout'] + 1e-5)
    out['total_skor_kognitif'] = df['skor_tryout'] + df['skor_literasi'] + df['skor_minat_belajar']
    
    return out

X_train_ultra = make_features_ultra(train)
X_test_ultra  = make_features_ultra(test)

cat_ultra = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.015,
    depth=6,
    l2_leaf_reg=7,
    bagging_temperature=0.8,
    random_seed=42,
    verbose=0
)

skor_ultra = cross_val_score(cat_ultra, X_train_ultra, y, cv=3).mean()
print(f"Akurasi Ultra Improve: {skor_ultra * 100:.2f}%")

Akurasi Ultra Improve: 52.16%


In [49]:
cat_ultra.fit(X_train_ultra, y)
pred_ultra = cat_ultra.predict(X_test_ultra)

submission_ultra = pd.DataFrame({
    "id": test["id"], 
    "target": pred_ultra.flatten()
})

submission_ultra.to_csv("submission_final.csv", index=False)
print("submission_final.csv BERHASIL DI-UPGRADE MAKSIMAL!")
submission_ultra.head()

submission_final.csv BERHASIL DI-UPGRADE MAKSIMAL!


,id,target
0,3,1
1,12,2
2,14,3
3,18,3
4,28,0


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

def make_features_stepwise(df):
    out = pd.DataFrame(index=df.index)
    
    wk = [f"nilai_minggu_{i:02d}" for i in range(1, 13)]
    dy = [f"aktivitas_hari_{i:02d}" for i in range(1, 17)]
    
    out['week_mean'] = df[wk].mean(axis=1)
    out['week_std'] = df[wk].std(axis=1)
    out['week_max'] = df[wk].max(axis=1)
    out['week_min'] = df[wk].min(axis=1)
    
    out['day_mean'] = df[dy].mean(axis=1)
    out['day_std'] = df[dy].std(axis=1)
    
    num_cols = ['skor_motivasi', 'skor_kedisiplinan', 'skor_tryout', 'skor_literasi', 'skor_minat_belajar', 'jarak_rumah_km']
    for c in num_cols:
        out[c] = df[c]
        
    out['tren_belajar'] = df['nilai_minggu_12'] - df['nilai_minggu_01']
    out['efisiensi_tugas'] = df['tugas_selesai'] / (df['tugas_diberikan'] + 1e-5)
    out['beban_vs_disiplin'] = df['tugas_diberikan'] * (df['skor_kedisiplinan'] + 6)
    out['total_kognitif'] = df['skor_tryout'] + df['skor_literasi'] + df['skor_minat_belajar']
    out['day_cv'] = out['day_std'] / (out['day_mean'] + 1e-5)
    out['rank_tryout'] = df['skor_tryout'].rank(pct=True)
    
    out['log_jarak'] = np.log1p(df['jarak_rumah_km'])
    
    return out

X_train_step = make_features_stepwise(train)
X_test_step  = make_features_stepwise(test)

cat_step = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.015,
    depth=6,
    random_seed=42,
    verbose=0
)

skor_step = cross_val_score(cat_step, X_train_step, y, cv=3).mean()
print(f"Hasil setelah perbaikan kolom: {skor_step * 100:.2f}%")

c:\Users\Pitri Wulandari\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Pitri Wulandari\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Pitri Wulandari\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Pitri Wulandari\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [54]:
print("Kolom yang ada di data kamu:")
print(train.columns.tolist())

Kolom yang ada di data kamu:
['id', 'nilai_minggu_01', 'nilai_minggu_02', 'nilai_minggu_03', 'nilai_minggu_04', 'nilai_minggu_05', 'nilai_minggu_06', 'nilai_minggu_07', 'nilai_minggu_08', 'nilai_minggu_09', 'nilai_minggu_10', 'nilai_minggu_11', 'nilai_minggu_12', 'skor_motivasi', 'skor_kedisiplinan', 'aktivitas_hari_01', 'aktivitas_hari_02', 'aktivitas_hari_03', 'aktivitas_hari_04', 'aktivitas_hari_05', 'aktivitas_hari_06', 'aktivitas_hari_07', 'aktivitas_hari_08', 'aktivitas_hari_09', 'aktivitas_hari_10', 'aktivitas_hari_11', 'aktivitas_hari_12', 'aktivitas_hari_13', 'aktivitas_hari_14', 'aktivitas_hari_15', 'aktivitas_hari_16', 'tugas_selesai', 'tugas_diberikan', 'kelas', 'urutan_ujian', 'skor_tryout', 'jarak_rumah_km', 'skor_ekstrakurikuler', 'indeks_kehadiran', 'skor_literasi', 'jumlah_saudara', 'skor_minat_belajar', 'target']
